In [0]:
%run ./phase2_common

In [0]:
MOVIES = f"{SCHEMA}.silver_movies_cleansed"

dbutils.widgets.text("batch_id", "", "Bronze batch ID")
batch_id = dbutils.widgets.get("batch_id").strip()
if not batch_id:
    raise ValueError("batch_id is required")

bronze = spark.table(BRONZE).filter(F.col("batch_id") == batch_id)
if not bronze.limit(1).count():
    raise ValueError(f"No Bronze records found for batch_id={batch_id}")
load_type = bronze.select("load_type").first()[0]

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {MOVIES} (
    movie_id BIGINT, imdb_id STRING, title STRING, original_title STRING,
    overview STRING, original_language STRING, release_date DATE, runtime INT,
    status STRING, adult BOOLEAN, video BOOLEAN, popularity DOUBLE,
    vote_average DOUBLE, vote_count BIGINT, budget BIGINT, revenue BIGINT,
    profit BIGINT, roi DOUBLE, budget_tier STRING, source_batch_id STRING,
    source_file STRING, source_extracted_at TIMESTAMP, row_hash STRING,
    load_timestamp TIMESTAMP
) USING DELTA
""")

In [0]:
parsed = (
    bronze.select(
        F.col("movie_id").alias("bronze_movie_id"), "batch_id", "source_file",
        "source_extracted_at", F.from_json("raw_payload", MOVIE_SCHEMA).alias("movie"),
    )
    .select("bronze_movie_id", "batch_id", "source_file", "source_extracted_at", "movie.*")
)

validated = with_validation_reason(parsed)
invalid = validated.filter(F.col("validation_reason").isNotNull())
valid = validated.filter(F.col("validation_reason").isNull())

budget = F.when(F.col("budget") > 0, F.col("budget"))
revenue = F.when(F.col("revenue") > 0, F.col("revenue"))
has_both = budget.isNotNull() & revenue.isNotNull()

business_columns = [
    "movie_id", "imdb_id", "title", "original_title", "overview", "original_language",
    "release_date", "runtime", "status", "adult", "video", "popularity", "vote_average",
    "vote_count", "budget", "revenue", "profit", "roi", "budget_tier",
]

source = (
    valid.select(
        F.col("id").alias("movie_id"),
        "imdb_id", "title", "original_title", "overview", "original_language",
        F.expr("try_to_date(NULLIF(TRIM(release_date), ''), 'yyyy-MM-dd')").alias("release_date"),
        "runtime", "status", "adult", "video", "popularity", "vote_average", "vote_count",
        budget.alias("budget"),
        revenue.alias("revenue"),
        F.when(has_both, revenue - budget).alias("profit"),
        F.when(has_both, (revenue - budget) / budget).alias("roi"),
        F.when(budget < 10_000_000, "low")
        .when(budget < 100_000_000, "medium")
        .when(budget.isNotNull(), "high")
        .alias("budget_tier"),
        F.col("batch_id").alias("source_batch_id"),
        "source_file", "source_extracted_at",
    )
    .withColumn("row_hash", F.sha2(F.to_json(F.struct(*business_columns)), 256))
    .withColumn("load_timestamp", F.current_timestamp())
)

In [0]:
def load():
    quarantined = invalid.count()
    merge_quarantine(invalid)

    if not source.limit(1).count():
        raise ValueError(f"No valid movies to load for batch_id={batch_id}")
    if source.groupBy("movie_id").count().filter("count > 1").limit(1).count():
        raise ValueError("Duplicate movie_id values in the Silver source")
    if source.filter(F.col("source_extracted_at").isNull()).limit(1).count():
        raise ValueError("Rows are missing source_extracted_at")

    stored = spark.table(MOVIES).select(
        "movie_id", F.col("source_extracted_at").alias("old_ts"), F.col("source_batch_id").alias("old_batch"),
    )
    older = F.col("source_extracted_at") < F.col("old_ts")
    same_time_other_batch = (F.col("source_extracted_at") == F.col("old_ts")) & (
        F.col("source_batch_id") != F.col("old_batch")
    )
    if source.join(stored, "movie_id").filter(older | same_time_other_batch).limit(1).count():
        raise ValueError("Snapshots are older than, or tied with, stored data; merge stopped")

    return {**upsert(MOVIES, source, ["movie_id"]), "quarantined": quarantined}


print(run_logged(
    "Bronze-to-Silver", load_type, f"batch_id={batch_id}; table=silver_movies_cleansed", batch_id, load,
))